# Notebook 07 — Model Comparison: LOO, WAIC, and Bayes Factors

## Background

Building a Bayesian model is only half the work. The other half is comparing it to alternatives: Is the hierarchical model actually better than the simpler version? Does adding a quadratic term improve fit enough to justify the complexity? Does the Student-t likelihood beat the Normal when data has heavy tails?

These questions require principled model comparison. In the frequentist world, you might use AIC or BIC. In the Bayesian world, the primary tools are:

- **LOO-CV** (Leave-One-Out Cross-Validation): estimate how well the model predicts new data by jackknifing. Implemented efficiently via PSIS-LOO in ArviZ.
- **WAIC** (Widely Applicable Information Criterion): a fully Bayesian generalization of AIC. Asymptotically equivalent to LOO-CV but faster.
- **Bayes factors**: ratio of marginal likelihoods `P(data | M1) / P(data | M2)`. Theoretically elegant but computationally intractable for most real models — often misused.

The current best practice (Vehtari, Gelman, Gabry 2017) is to use **PSIS-LOO** as the primary comparison tool, with WAIC as a cross-check. Bayes factors are mentioned for completeness but rarely used in practice with complex models.

## What You'll Learn

- LOO-CV: the leave-one-out predictive density as a model quality measure
- PSIS-LOO: Pareto-smoothed importance sampling — efficient LOO without refitting
- WAIC: log pointwise predictive density with an effective-parameters penalty
- `az.compare()`: ranking multiple models in ArviZ
- When LOO disagrees with your intuitions — and what to do about it
- Bayes factors: when they work and when they don't

## Why This Matters

Without model comparison, you're flying blind. A more complex model always fits training data better — the question is whether that complexity reflects genuine structure or overfitting. LOO-CV answers this by estimating predictive accuracy on held-out data. In the Bayesian framework, this estimate also propagates parameter uncertainty — it's not just a point estimate of out-of-sample accuracy, it's a full assessment.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — The Problem with Log-Likelihood Comparisons

The simplest model comparison is to compare log-likelihoods: which model assigns higher probability to the observed data? But log-likelihood always increases with model complexity — adding parameters can only help (or stay the same) in-sample. The question is whether the added complexity helps *out of sample*.

Information criteria (AIC, BIC, WAIC) penalize complexity. LOO-CV directly estimates out-of-sample performance. Both approaches ask the same question via different mechanisms.

In [ ]:
np.random.seed(42)

# Dataset: polynomial regression comparison
# True relationship is cubic; we'll compare linear, quadratic, cubic, and degree-5 fits
n = 60
x = np.linspace(-2, 2, n)
true_y = 0.5 * x**3 - x + np.random.normal(0, 0.5, n)

x_z = (x - x.mean()) / x.std()

# Pre-compute polynomial features
X = {d: np.stack([x_z**i for i in range(d+1)], axis=1)
     for d in [1, 2, 3, 5]}

traces = {}

for degree in [1, 2, 3, 5]:
    with pm.Model() as poly_model:
        # Priors on each polynomial coefficient
        coeffs = pm.Normal('coeffs', mu=0, sigma=3, shape=degree+1)
        sigma  = pm.HalfNormal('sigma', sigma=2)
        mu     = X[degree] @ coeffs
        y_obs  = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=true_y)
        trace  = pm.sample(draws=1000, tune=500, chains=2,
                           random_seed=42, progressbar=False)
        # Compute log-likelihood for LOO/WAIC
        pm.compute_log_likelihood(trace)
        traces[degree] = (trace, poly_model)

print('Models fitted: degree 1, 2, 3, 5')
print('Next: compute LOO-CV and WAIC for each')

In [ ]:
# Compute LOO-CV for all models using PSIS-LOO (Vehtari 2017)
loo_results = {}
for degree, (trace, _) in traces.items():
    loo_results[degree] = az.loo(trace, pointwise=True)
    print(f'Degree {degree}: LOO={loo_results[degree].elpd_loo:.2f} +/- {loo_results[degree].se:.2f}  '
          f'  p_loo={loo_results[degree].p_loo:.2f}')

In [ ]:
# az.compare: ranks models by LOO-CV; handles uncertainty in the comparison
# Note: az.compare expects InferenceData objects
compare_dict = {f'degree_{d}': traces[d][0] for d in [1, 2, 3, 5]}
comparison = az.compare(compare_dict, ic='loo', scale='log')
print('Model Comparison (LOO-CV, higher is better):')
print(comparison.to_string())
print()
print('Columns:')
print('  elpd_loo: expected log pointwise predictive density (higher = better)')
print('  p_loo:    effective number of parameters (higher = more complex)')
print('  d_loo:    difference from best model')
print('  weight:   model weight for stacking ensemble')

In [ ]:
# ArviZ built-in comparison plot
az.plot_compare(comparison, figsize=(9, 4))
plt.title('LOO-CV Model Comparison\nDot = LOO estimate, bar = uncertainty')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  - If models overlap in LOO, the difference is not meaningful')
print('  - degree_3 should be preferred (true model has cubic terms)')
print('  - degree_5 may overfit despite fitting training data better')

In [ ]:
# Visualize what each polynomial model fits
x_plot = np.linspace(-2.2, 2.2, 200)
x_plot_z = (x_plot - x.mean()) / x.std()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
colors = ['darkorange', 'green', 'steelblue', 'red']

for ax, (degree, (trace, _), color) in zip(axes.flatten(),
                                             [(d, traces[d], c) for d, c in zip([1,2,3,5], colors)]):
    coeffs_samp = trace.posterior['coeffs'].values.reshape(-1, degree+1)
    X_plot = np.stack([x_plot_z**i for i in range(degree+1)], axis=1)
    
    # Posterior predictive curves
    mu_curves = (coeffs_samp @ X_plot.T)  # (samples, n_plot)
    mu_mean = mu_curves.mean(axis=0)
    mu_lo, mu_hi = np.percentile(mu_curves, [3, 97], axis=0)
    
    ax.scatter(x, true_y, alpha=0.4, s=20, color='gray', label='Data')
    ax.fill_between(x_plot, mu_lo, mu_hi, alpha=0.25, color=color)
    ax.plot(x_plot, mu_mean, color=color, lw=2, label='Posterior mean')
    ax.plot(x_plot, 0.5*x_plot**3 - x_plot, 'k--', lw=1.5, alpha=0.8, label='True')
    loo_val = loo_results[degree].elpd_loo
    ax.set_title(f'Degree {degree}  |  LOO={loo_val:.1f}', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_ylim(-5, 5)

plt.suptitle('Polynomial Regression: Fit vs LOO-CV Score', fontsize=12)
plt.tight_layout()
plt.show()

## Part 2 — WAIC: The Information-Theoretic Approach

WAIC (Watanabe 2010) estimates the expected log pointwise predictive density (elpd) using the posterior:

```
WAIC = -2 * (LPPD - p_WAIC)
```

Where:
- `LPPD = sum_i log(mean_s P(y_i | theta_s))` — log pointwise predictive density averaged over posterior samples
- `p_WAIC = sum_i var_s(log P(y_i | theta_s))` — effective number of parameters (variance of per-observation log-likelihoods)

WAIC is asymptotically equivalent to LOO-CV but uses the full posterior rather than leaving one observation out at a time. It's faster but less reliable than LOO for small datasets or when individual observations have high influence.

In [ ]:
# WAIC for same models
waic_results = {}
for degree, (trace, _) in traces.items():
    waic_results[degree] = az.waic(trace, pointwise=True)
    print(f'Degree {degree}: WAIC={waic_results[degree].elpd_waic:.2f}  '
          f'p_waic={waic_results[degree].p_waic:.2f}')

print()
# Compare LOO vs WAIC agreement
print('LOO vs WAIC comparison:')
print(f'{"Degree":>8} {"LOO":>8} {"WAIC":>8} {"Diff":>8}')
print('-' * 36)
for d in [1, 2, 3, 5]:
    loo_val  = loo_results[d].elpd_loo
    waic_val = waic_results[d].elpd_waic
    print(f'{d:>8} {loo_val:>8.2f} {waic_val:>8.2f} {abs(loo_val-waic_val):>8.2f}')
print('\nWAIC and LOO should agree closely for well-behaved models.')

## Part 3 — Pareto k Diagnostics

PSIS-LOO approximates the leave-one-out predictive density using importance weights. The reliability of this approximation is measured by the *Pareto k* statistic for each data point. When `k > 0.7`, the importance weights for that observation are too extreme — the LOO estimate is unreliable for that point.

High `k` values indicate *influential observations* — points that disproportionately affect the fitted model. They deserve investigation: are they outliers? Errors? Genuinely surprising data points?

In [ ]:
# Generate data with some genuinely influential outliers
np.random.seed(42)
n_loo = 50
x_loo = np.random.normal(0, 1, n_loo)
y_loo = 2 * x_loo + np.random.normal(0, 1, n_loo)

# Add a few leverage points
x_loo = np.append(x_loo, [4.0, 4.5, -4.0])   # extreme x values
y_loo = np.append(y_loo, [0.0, 0.5, -0.5])    # but unusual y given x
n_loo = len(x_loo)
x_z_loo = (x_loo - x_loo.mean()) / x_loo.std()

with pm.Model() as loo_diag_model:
    alpha_p = pm.Normal('alpha', mu=0, sigma=5)
    beta_p  = pm.Normal('beta',  mu=0, sigma=3)
    sigma_p = pm.HalfNormal('sigma', sigma=2)
    mu_p    = alpha_p + beta_p * x_z_loo
    y_obs_p = pm.Normal('y_obs', mu=mu_p, sigma=sigma_p, observed=y_loo)
    trace_loo_diag = pm.sample(draws=1000, tune=500, chains=2,
                                random_seed=42, progressbar=False)
    pm.compute_log_likelihood(trace_loo_diag)

loo_diag = az.loo(trace_loo_diag, pointwise=True)
k_values = loo_diag.pareto_k

print(f'Pareto k diagnostics:')
print(f'  k < 0.5 (good):     {(k_values < 0.5).sum():3d} points')
print(f'  0.5 <= k < 0.7:     {((k_values >= 0.5) & (k_values < 0.7)).sum():3d} points')
print(f'  k >= 0.7 (bad):     {(k_values >= 0.7).sum():3d} points  <- investigate these!')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pareto k plot
ax = axes[0]
colors_k = ['green' if k < 0.5 else ('orange' if k < 0.7 else 'red') for k in k_values]
ax.scatter(np.arange(n_loo), k_values, c=colors_k, s=40, alpha=0.8)
ax.axhline(0.5, color='orange', linestyle='--', lw=1.5, label='k=0.5 (marginal)')
ax.axhline(0.7, color='red',    linestyle='--', lw=1.5, label='k=0.7 (problematic)')
ax.set_xlabel('Observation index'); ax.set_ylabel('Pareto k')
ax.set_title('PSIS-LOO Pareto k Diagnostics\nHigh k = influential observations')
ax.legend()

# Scatter plot: highlight problematic observations
ax = axes[1]
ax.scatter(x_loo[:n_loo-3], y_loo[:n_loo-3], color='steelblue', alpha=0.5, s=30, label='Normal')
for i in range(n_loo-3, n_loo):
    ax.scatter(x_loo[i], y_loo[i], color='red', s=100, zorder=5,
               label=f'Leverage point (k={k_values[i]:.2f})' if i == n_loo-3 else '')

# Fitted line
x_fit = np.linspace(x_loo.min()-0.3, x_loo.max()+0.3, 100)
x_fit_z = (x_fit - x_loo.mean()) / x_loo.std()
alpha_fit = float(trace_loo_diag.posterior['alpha'].values.mean())
beta_fit  = float(trace_loo_diag.posterior['beta'].values.mean())
ax.plot(x_fit, alpha_fit + beta_fit * x_fit_z, 'steelblue', lw=2, label='Fitted')
ax.plot(x_fit, 2 * x_fit, 'r--', lw=1.5, alpha=0.7, label='True')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Leverage Points Pull the Fit\nRed = high-k (influential) observations')
ax.legend(fontsize=8)

plt.suptitle('Pareto k Diagnostics: Finding Influential Observations', fontsize=12)
plt.tight_layout()
plt.show()

## Part 4 — Practical Model Comparison: Normal vs Student-t

A realistic comparison: should we use a Normal or Student-t likelihood? The data has some outliers. We'll use LOO-CV to quantify the predictive accuracy difference.

In [ ]:
np.random.seed(42)

# Data with a few genuine outliers
n_comp = 80
x_comp = np.random.uniform(-3, 3, n_comp)
y_comp = 2 * x_comp + np.random.normal(0, 1, n_comp)
# Add outliers
outlier_idx = np.random.choice(n_comp, 6, replace=False)
y_comp[outlier_idx] += np.random.choice([-6, 6], 6) * np.random.uniform(1, 2, 6)

x_z_comp = (x_comp - x_comp.mean()) / x_comp.std()

# Model A: Normal likelihood
with pm.Model() as model_normal:
    alpha = pm.Normal('alpha', mu=0, sigma=5)
    beta  = pm.Normal('beta',  mu=0, sigma=3)
    sigma = pm.HalfNormal('sigma', sigma=5)
    y_obs = pm.Normal('y_obs', mu=alpha + beta * x_z_comp, sigma=sigma, observed=y_comp)
    trace_normal = pm.sample(draws=2000, tune=1000, chains=4,
                              random_seed=42, progressbar=False)
    pm.compute_log_likelihood(trace_normal)

# Model B: Student-t likelihood
with pm.Model() as model_student:
    alpha = pm.Normal('alpha', mu=0, sigma=5)
    beta  = pm.Normal('beta',  mu=0, sigma=3)
    sigma = pm.HalfNormal('sigma', sigma=5)
    nu    = pm.Exponential('nu', lam=1/30)
    y_obs = pm.StudentT('y_obs', nu=nu, mu=alpha + beta * x_z_comp,
                         sigma=sigma, observed=y_comp)
    trace_student = pm.sample(draws=2000, tune=1000, chains=4,
                               random_seed=42, progressbar=False)
    pm.compute_log_likelihood(trace_student)

# Compare
comparison_ns = az.compare(
    {'Normal': trace_normal, 'StudentT': trace_student},
    ic='loo', scale='log'
)
print('Normal vs Student-t Likelihood:')
print(comparison_ns.to_string())

In [ ]:
az.plot_compare(comparison_ns, figsize=(8, 3))
plt.title('LOO-CV: Normal vs Student-t Likelihood')
plt.tight_layout()
plt.show()

# Show how the two models handle outliers differently
alpha_n = trace_normal.posterior['alpha'].values.flatten()
beta_n  = trace_normal.posterior['beta'].values.flatten()
alpha_t = trace_student.posterior['alpha'].values.flatten()
beta_t  = trace_student.posterior['beta'].values.flatten()

x_fit_c = np.linspace(-3.5, 3.5, 200)
x_fit_z = (x_fit_c - x_comp.mean()) / x_comp.std()

mu_n = alpha_n.mean() + beta_n.mean() * x_fit_z
mu_t = alpha_t.mean() + beta_t.mean() * x_fit_z

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_comp, y_comp, alpha=0.4, s=25, color='gray', label='Data')
ax.scatter(x_comp[outlier_idx], y_comp[outlier_idx], color='red', s=80, zorder=5,
           marker='x', lw=2, label='Outliers')
ax.plot(x_fit_c, mu_n, 'darkorange', lw=2, label='Normal likelihood')
ax.plot(x_fit_c, mu_t, 'steelblue',  lw=2, label='Student-t likelihood')
ax.plot(x_fit_c, 2 * x_fit_c, 'k--', lw=1.5, alpha=0.7, label='True')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Normal vs Student-t: Effect on Fitted Line\nStudent-t downweights outliers')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

nu_samp = trace_student.posterior['nu'].values.flatten()
print(f'Posterior nu (degrees of freedom): {nu_samp.mean():.1f} (94% CI: {np.percentile(nu_samp, 3):.1f}, {np.percentile(nu_samp, 97):.1f})')
print('Low nu -> model detected heavy tails / outliers')

## Part 5 — Bayes Factors: When They Work and When They Don't

The Bayes factor `BF_{12} = P(data | M1) / P(data | M2)` is the ratio of marginal likelihoods. It updates the prior odds to the posterior odds:

```
P(M1|data)   BF_{12}   P(M1)
────────── = ──────── × ─────
P(M2|data)     1       P(M2)
```

**When Bayes factors work**: comparing simple models with analytically tractable marginal likelihoods (e.g., conjugate models, point-null hypothesis testing in low dimensions).

**When they don't work** (most practical situations):
- The marginal likelihood `P(data|M) = ∫ P(data|theta, M) P(theta|M) dtheta` is an extremely high-dimensional integral — typically intractable
- Bayes factors are **extremely sensitive to the prior**: proper, diffuse priors can make `P(data|M)` arbitrarily small even for a correct model
- Computing them requires either conjugate models or expensive special-purpose algorithms (bridge sampling, thermodynamic integration)

For model comparison in practice, **LOO-CV is almost always preferred** over Bayes factors for non-conjugate models.

In [ ]:
# Demonstrate Bayes factor computation for a simple conjugate case
# Beta-Binomial: H0 (theta=0.5 exactly) vs H1 (theta ~ Beta(1,1))
# This is tractable because both marginal likelihoods have closed forms

def beta_binomial_marginal_likelihood(k, n, alpha, beta):
    """P(k | n, Beta(alpha, beta)) = C(n,k) * B(k+alpha, n-k+beta) / B(alpha, beta)"""
    from scipy.special import betaln, comb
    log_C = np.log(comb(n, k, exact=True))
    log_B_post  = betaln(k + alpha, n - k + beta)
    log_B_prior = betaln(alpha, beta)
    return np.exp(log_C + log_B_post - log_B_prior)

def null_likelihood(k, n, theta0=0.5):
    """P(k | n, theta=theta0)"""
    return stats.binom.pmf(k, n, theta0)

# Simulate: is a coin fair?
np.random.seed(42)
coin_results = []

print('Bayesian Hypothesis Testing: Is the Coin Fair?')
print('H0: theta = 0.5 exactly')
print('H1: theta ~ Beta(1, 1) (any probability equally likely)')
print()
print(f'{"n":>5} {"k":>5} {"k/n":>6} {"BF_10":>10} {"Evidence"}' )
print('-' * 55)

for n_tosses in [10, 20, 50, 100, 200]:
    k = int(0.65 * n_tosses)  # always 65% heads
    ml_h1 = beta_binomial_marginal_likelihood(k, n_tosses, 1, 1)
    ml_h0 = null_likelihood(k, n_tosses)
    bf = ml_h1 / ml_h0
    
    if bf > 100:
        strength = 'Decisive for H1'
    elif bf > 10:
        strength = 'Strong for H1'
    elif bf > 3:
        strength = 'Moderate for H1'
    elif bf > 1:
        strength = 'Anecdotal for H1'
    elif bf > 1/3:
        strength = 'Anecdotal for H0'
    else:
        strength = 'Strong for H0'
    
    print(f'{n_tosses:>5} {k:>5} {k/n_tosses:>6.2f} {bf:>10.2f}  {strength}')

print()
print('Bayes factors accumulate evidence monotonically with data.')
print('For non-conjugate models, use LOO-CV instead.')

In [ ]:
# Kass & Raftery (1995) Bayes Factor interpretation table
print('Kass & Raftery (1995) Bayes Factor Guidelines:')
print('=' * 50)
table = [
    ('BF_10 < 1',     'Evidence for H0 (H1 disfavored)'),
    ('1 - 3',         'Anecdotal evidence for H1'),
    ('3 - 10',        'Moderate evidence for H1'),
    ('10 - 30',       'Strong evidence for H1'),
    ('30 - 100',      'Very strong evidence for H1'),
    ('> 100',         'Decisive evidence for H1'),
]
for bf_range, label in table:
    print(f'  {bf_range:20} {label}')

print()
print('Practical note: Bayes factors require PROPER priors.')
print('Improper priors (like Jeffreys) make BF undefined.')
print('Vague proper priors (Normal(0, 1000)) artificially inflate evidence for H0.')
print('For complex models: use LOO-CV or WAIC instead.')

## Part 6 — Model Stacking: Beyond Selecting One Model

LOO-CV gives you model weights for stacking: a weighted ensemble that optimizes predictive accuracy. Rather than picking one "winner", stacking combines predictions from all models, weighted by their out-of-sample performance.

In [ ]:
# The 'weight' column in az.compare already gives stacking weights
print('Model Stacking Weights (from LOO-CV):')
print(comparison.to_string())
print()
print('Stacking interpretation:')
print('  weight=1.0 for best model: LOO says use it exclusively')
print('  weight split across models: combining improves predictions')
print()
print('az.compare gives Bayesian model averaging weights by default.')
print('Use method="stacking" for stacking (optimizes predictive accuracy directly).')

# Recompute with stacking method
comparison_stacking = az.compare(compare_dict, ic='loo', method='stacking')
print('\nWith stacking method:')
print(comparison_stacking[['elpd_loo', 'weight']].to_string())

## Summary

**Model comparison hierarchy** (most to least preferred in practice):

1. **PSIS-LOO** (`az.loo` + `az.compare`): best for most models; reliable, fast, provides Pareto k diagnostics
2. **WAIC** (`az.waic`): asymptotically equivalent to LOO, faster, but less reliable for small n or influential observations
3. **Bayes factors**: only for simple conjugate models; computationally intractable and prior-sensitive otherwise

**Workflow**:
```python
# After sampling each model, compute log likelihood:
with model:
    pm.compute_log_likelihood(trace)

# Compare:
comparison = az.compare({'model_A': trace_a, 'model_B': trace_b}, ic='loo')
az.plot_compare(comparison)

# Check influential points:
loo_result = az.loo(trace, pointwise=True)
# loo_result.pareto_k -- values > 0.7 indicate problematic observations
```

**Key insight**: LOO-CV difference is only meaningful if larger than its standard error. Models within ~2-4 elpd units of each other are effectively equivalent for predictive purposes — prefer the simpler one.

**Next**: Notebook 08 — Gaussian Processes. Bayesian priors over functions; GP regression and classification in PyMC.